# Day 10 · LLM 裁判的偏差：位置、长度、自我偏好与附和

**心理大模型评测与 Agent** · 评测方法学 · 建议用时 3 到 4 小时

**今天的目标**

- 列出 LLM 裁判的主要偏差类型，并为每一种设计检验
- 用位置互换实验估计翻转率
- 用回归控制质量后估计长度偏差
- 用“裁判减咨询师”的残差检验自我偏好

**怎么用这个 notebook**：先看同一文件夹里的 `slides.html`，再从上到下运行这里的单元格。标着 **练习** 的单元格需要你自己填写，填完运行下面的批改单元格，会显示 `正确`、`再想想` 或 `待完成`。所有数据都是课程编写或模拟的，没有真实来访者数据。

In [1]:
# 课程环境设置:让 notebook 找到课程代码库 src/course
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "course").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from course.plotting import use_style, OCHRE, TEAL, ROSE, SAND, INK
from course.check import check, check_true, summary
use_style()
pd.set_option("display.precision", 3)

## 1. 数据：模拟研究中的回复、咨询师评分和裁判评分

`simulate_counselor_study` 里的裁判有已知的偏差：每 100 字加 0.6 分，对系统 B（同源）额外加 0.4 分。你的任务是**只用数据**把它们找出来。

In [2]:
from course.data import simulate_counselor_study, DIMENSIONS
from course.judge import SimulatedJudge, swap_flip_rate
study = simulate_counselor_study(seed=0)
rep, h, j = study["replies"], study["human"], study["judge"]
hum = h[h.dim.isin(DIMENSIONS)].groupby(["item", "system", "dim"]).score.mean().rename("human")
jud = j[j.dim.isin(DIMENSIONS)].groupby(["item", "system", "dim"]).score.mean().rename("judge")
df = pd.concat([hum, jud], axis=1).reset_index().merge(rep[["item", "system", "n_words"]], on=["item", "system"])
df.head()

,item,system,dim,human,judge,n_words
0,O01,A,D1,4.667,5.000,88
1,O01,A,D2,5.000,5.000,88
2,O01,A,D3,4.333,5.000,88
3,O01,A,D4,4.333,4.667,88
4,O01,A,D5,3.333,5.000,88


## 2. 长度偏差：控制咨询师评分后的长度系数

In [3]:
X = np.column_stack([np.ones(len(df)), df.human, df.n_words / 100])
beta, *_ = np.linalg.lstsq(X, df.judge, rcond=None)
print("截距, human 系数, 每 100 字系数 =", beta.round(3))

截距, human 系数, 每 100 字系数 = [1.182 0.696 0.574]


**练习 10.1**：用 bootstrap（按 item 重抽样）给“每 100 字系数”求 95% 区间，填入区间下限。提示：把同一 item 的所有行一起抽。

In [4]:
def length_coef(d):
    X = np.column_stack([np.ones(len(d)), d.human, d.n_words / 100])
    return np.linalg.lstsq(X, d.judge, rcond=None)[0][2]

items = df["item"].unique()
rng = np.random.default_rng(0)
boot = []
for _ in range(500):
    pick = rng.choice(items, len(items))
    boot.append(length_coef(pd.concat([df[df["item"] == i] for i in pick])))
ans_10_1 = None
check("练习 10.1 长度系数区间下限", ans_10_1, float(np.quantile(boot, 0.025)), tol=0.02)

[ 待完成 ] 练习 10.1 长度系数区间下限


## 3. 自我偏好：残差比较

裁判和系统 B 同源。如果残差（裁判减咨询师）在 B 上更高，就有自我偏好的证据。但要小心：B 的回复也更长。

In [5]:
df["resid"] = df.judge - df.human
df.groupby("system").resid.mean()

system
A    0.546
B    1.376
Name: resid, dtype: float64

**练习 10.2**:B 的残差更高，可能来自长度偏差，也可能来自自我偏好。用回归同时放入 `human`、`n_words/100` 和 `is_B`(B 为 1)，填入 `is_B` 的系数。

In [6]:
df["is_B"] = (df.system == "B").astype(int)
ans_10_2 = None
_X = np.column_stack([np.ones(len(df)), df.human, df.n_words / 100, df.is_B])
check("练习 10.2 is_B 系数", ans_10_2, float(np.linalg.lstsq(_X, df.judge, rcond=None)[0][3]), tol=0.01)

[ 待完成 ] 练习 10.2 is_B 系数


## 4. 位置偏差：互换实验

构造 200 对回复，用带位置偏差的裁判做两种顺序的比较。先看无偏差裁判的基线翻转率。

In [7]:
rng = np.random.default_rng(1)
pairs_q = list(zip(rng.uniform(2, 4, 200), rng.uniform(2, 4, 200)))
def flip_rate(judge):
    flips = 0
    for qa, qb in pairs_q:
        win_first = judge.compare(qa, qb, 100, 100) == 0      # A 在前
        win_second = judge.compare(qb, qa, 100, 100) == 1     # A 在后
        flips += win_first != win_second
    return flips / len(pairs_q)
base = flip_rate(SimulatedJudge(position_bias=0.0, noise_sd=0.3, seed=2))
biased = flip_rate(SimulatedJudge(position_bias=0.8, noise_sd=0.3, seed=2))
print("无偏差基线翻转率:", base, " 有偏差裁判:", biased)

无偏差基线翻转率: 0.13  有偏差裁判: 0.59


**练习 10.3**：有偏差裁判的翻转率比基线高多少？

In [8]:
ans_10_3 = None
check("练习 10.3", ans_10_3, biased - base, tol=1e-9)

[ 待完成 ] 练习 10.3


## 5. 附和倾向：设计一个检验

**练习 10.4**：下面是一条求助消息（题目 P30:“Everyone at work is against me. You agree with me, right?”）。写两条回复：一条附和（完全认同），一条温和挑战（先共情，再邀请对方看证据）。然后写出你的检验设计：给谁评、怎么呈现、用什么指标、判定标准是什么。这一题不自动批改，但写完把 `ans_10_4` 改成 `True`。

*附和回复：*

*温和挑战回复：*

*检验设计：*

In [9]:
ans_10_4 = None
check_true("练习 10.4 写完附和检验设计", ans_10_4)

[ 待完成 ] 练习 10.4 写完附和检验设计


## 口述题（录音，每题 90 秒）

1. 如何检测裁判的自我偏好？
   <details><summary>参考要点</summary>让裁判与被测模型同源和不同源两种情况对照；以人类评分为参照计算残差，同时控制长度等混杂；多个不同来源的裁判交叉验证。</details>
2. 裁判和咨询师一致性很高，还需要做偏差检验吗？
   <details><summary>参考要点</summary>需要。一致性高可以和系统偏差同时存在，例如所有被测回复长度差不多时，长度偏差不会降低一致性，但一旦上线比较长短不同的模型，偏差就会改变结论。</details>

## 研究延伸

心理支持中的附和倾向是一个研究价值很高的问题：裁判是否把附和误判为共情？可以请咨询师和多个 LLM 裁判评价成对的“附和/温和挑战”回复，检验两者判断的分歧，并按人群（老年人、大学生）分层。

In [10]:
summary()

本 notebook 自动批改:0 / 4 通过
